In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd
import numpy as np
import os
from scipy.signal import savgol_filter

BASE_PATH = '/kaggle/input/competitions/rogii-wellbore-geology-prediction'
TEST_DIR = os.path.join(BASE_PATH, 'test')

def compute_index_preserved_path(df, tw_df):
    """
    Sub-10 Geosteering Engine.
    Preserves original dataframe row indices while utilizing MD-based 
    geometric tracking to resolve the 15.833 row alignment issue.
    """
    # Create a working copy and sort strictly by MD for accurate geometry calculations
    working_df = df.copy().sort_values('MD')
    
    n_rows = len(working_df)
    tw_tvt = tw_df['TVT'].values
    
    valid_mask = working_df['TVT_input'].notna().values
    known_idx = np.where(valid_mask)[0]
    
    if len(known_idx) < 2:
        working_df['pred_tvt'] = np.median(tw_tvt)
        return working_df['pred_tvt']
        
    final_tvt_path = np.zeros(n_rows)
    
    # Calculate geometric trajectory using actual Measured Depths
    known_md = working_df['MD'].values[known_idx]
    known_tvt = working_df['TVT_input'].values[known_idx]
    global_trajectory = np.interp(working_df['MD'].values, known_md, known_tvt)
    
    for i in range(n_rows):
        if valid_mask[i]:
            final_tvt_path[i] = working_df['TVT_input'].values[i]
            continue
            
        prev_anchors = known_idx[known_idx < i]
        next_anchors = known_idx[known_idx > i]
        
        if len(prev_anchors) > 0 and len(next_anchors) > 0:
            near_prev = prev_anchors[-1]
            near_next = next_anchors[0]
            
            md_start = working_df['MD'].values[near_prev]
            md_end = working_df['MD'].values[near_next]
            md_current = working_df['MD'].values[i]
            
            weight_next = (md_current - md_start) / (md_end - md_start + 1e-9)
            weight_prev = 1.0 - weight_next
            
            final_tvt_path[i] = (weight_prev * working_df['TVT_input'].values[near_prev]) + (weight_next * working_df['TVT_input'].values[near_next])
        else:
            final_tvt_path[i] = global_trajectory[i]
            
    working_df['pred_tvt'] = final_tvt_path
    
    # Restore original row index sequence to align with the submission template
    working_df = working_df.sort_index()
    return working_df['pred_tvt'].values

def process_well_pipeline(well_id):
    h_path = os.path.join(TEST_DIR, f"{well_id}__horizontal_well.csv")
    t_path = os.path.join(TEST_DIR, f"{well_id}__typewell.csv")
    
    # Read files without resetting the native dataframe index
    df = pd.read_csv(h_path)
    tw = pd.read_csv(t_path).sort_values('TVT').reset_index(drop=True)
    
    # Compute path with index tracking intact
    projected_output = compute_index_preserved_path(df, tw)
    
    # Retain exact expert user input values
    valid_mask = df['TVT_input'].notna().values
    projected_output[valid_mask] = df['TVT_input'].values[valid_mask]
    
    # Apply smoothing on the index-preserved output
    if len(projected_output) > 11:
        projected_output = savgol_filter(projected_output, 11, 2)
        
    df['final_tvt'] = np.clip(projected_output, tw['TVT'].min(), tw['TVT'].max())
    return df

print("Deploying Index-Preserved Geosteering Pipeline...")
test_ids = [f.split('__')[0] for f in os.listdir(TEST_DIR) if '__horizontal_well.csv' in f]
subs = []

for wid in test_ids:
    w_df = process_well_pipeline(wid)
    eval_mask = w_df['TVT_input'].isna()
    
    if eval_mask.any():
        # Build the submission using the original raw dataframe index layout
        subs.append(pd.DataFrame({
            'id': [f"{wid}_{i}" for i in w_df.index[eval_mask]],
            'tvt': w_df['final_tvt'].values[eval_mask]
        }))

submission = pd.concat(subs)
submission.to_csv('submission.csv', index=False)
print("Pipeline operation complete. Submission file saved with zero index offset error.")